In [1]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

--2026-09-15 01:25:48--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip.2’

ml-latest-small.zip 100%[===================>] 955.28K   764KB/s    in 1.3s    

2026-09-15 01:25:51 (764 KB/s) - ‘ml-latest-small.zip.2’ saved [978202/978202]



In [4]:
!unzip ml-latest-small.zip

Archive:  ml-latest-small.zip
replace ml-latest-small/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [7]:
import os

os.listdir("ml-latest-small")

['ratings.csv', 'links.csv', 'README.txt', 'movies.csv', 'tags.csv']

In [8]:
import pandas as pd

movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')
tags = pd.read_csv('ml-latest-small/tags.csv')

In [9]:
print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [10]:
# Har movie ke saare tags ko ek string mein jodo
tags_combined = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

# Movies ke saath merge karo
movies = movies.merge(tags_combined, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')

# Genre aur tags ko combine karo ek "features" column mein
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['features'] = movies['genres_clean'] + ' ' + movies['tag']

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['features'])
content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [12]:
!pip install scikit-surprise -q

from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
svd_model = SVD()
svd_model.fit(trainset)

predictions = svd_model.test(testset)
print("Model accuracy (RMSE):", accuracy.rmse(predictions))

RMSE: 0.8815
Model accuracy (RMSE): 0.8814967254673701


In [15]:
indices = pd.Series(movies.index, index=movies['title'].str.lower())

def hybrid_recommend(title, user_id=1, top_n=10):
    title = title.lower()
    if title not in indices:
        return f"'{title}' dataset mein nahi mili."

    idx = indices[title]

    # Content-based similarity scores
    sim_scores = list(enumerate(content_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:50]  # top 50 candidates content ke basis par

    movie_indices = [i[0] for i in sim_scores]
    candidates = movies.iloc[movie_indices].copy()
    candidates['content_score'] = [i[1] for i in sim_scores]

    # Collaborative filtering se predicted rating add karo
    candidates['predicted_rating'] = candidates['movieId'].apply(
        lambda x: svd_model.predict(user_id, x).est
    )

    # Final hybrid score = content similarity + predicted rating (normalized weight)
    candidates['hybrid_score'] = (candidates['content_score'] * 0.5) + \
                                   (candidates['predicted_rating'] / 5 * 0.5)

    candidates = candidates.sort_values('hybrid_score', ascending=False)

    return candidates[['title', 'content_score', 'predicted_rating', 'hybrid_score']].head(top_n)

In [14]:
print(hybrid_recommend('Toy Story (1995)', user_id=1))
print(hybrid_recommend('Batman Forever (1995)', user_id=15))

                                                  title  content_score  \
1757                               Bug's Life, A (1998)       0.862225   
2355                                 Toy Story 2 (1999)       0.644038   
8695                   Guardians of the Galaxy 2 (2017)       0.367650   
9430                                       Moana (2016)       0.357912   
3568                              Monsters, Inc. (2001)       0.357912   
6944                 Ponyo (Gake no ue no Ponyo) (2008)       0.344861   
7039                                          Up (2009)       0.320916   
1505                         Black Cauldron, The (1985)       0.344861   
7184                               Partly Cloudy (2009)       0.325317   
5624  Kirikou and the Sorceress (Kirikou et la sorci...       0.344861   

      predicted_rating  hybrid_score  
1757          4.385219      0.869634  
2355          4.827156      0.804735  
8695          4.632405      0.647066  
9430          4.671021      0

In [16]:
%%writefile app.py
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader

st.title("🎬 Hybrid Movie Recommendation System")

@st.cache_data
def load_data():
    movies = pd.read_csv('ml-latest-small/movies.csv')
    ratings = pd.read_csv('ml-latest-small/ratings.csv')
    tags = pd.read_csv('ml-latest-small/tags.csv')

    tags_combined = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
    movies = movies.merge(tags_combined, on='movieId', how='left')
    movies['tag'] = movies['tag'].fillna('')
    movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
    movies['features'] = movies['genres_clean'] + ' ' + movies['tag']

    return movies, ratings

@st.cache_resource
def build_models(movies, ratings):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movies['features'])
    content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
    trainset = data.build_full_trainset()
    svd_model = SVD()
    svd_model.fit(trainset)

    return content_sim, svd_model

movies, ratings = load_data()
content_sim, svd_model = build_models(movies, ratings)
indices = pd.Series(movies.index, index=movies['title'].str.lower())

def hybrid_recommend(title, user_id, top_n=10):
    title = title.lower()
    idx = indices[title]
    sim_scores = list(enumerate(content_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:50]
    movie_indices = [i[0] for i in sim_scores]
    candidates = movies.iloc[movie_indices].copy()
    candidates['content_score'] = [i[1] for i in sim_scores]
    candidates['predicted_rating'] = candidates['movieId'].apply(
        lambda x: svd_model.predict(user_id, x).est
    )
    candidates['hybrid_score'] = (candidates['content_score'] * 0.5) + \
                                   (candidates['predicted_rating'] / 5 * 0.5)
    candidates = candidates.sort_values('hybrid_score', ascending=False)
    return candidates[['title', 'predicted_rating']].head(top_n)

st.write("Movie choose karo aur apna User ID daalo - personalized recommendations milenge!")

movie_list = movies['title'].values
selected_movie = st.selectbox("Movie select karo:", movie_list)
user_id = st.number_input("User ID daalo:", min_value=1, max_value=int(ratings['userId'].max()), value=1)

if st.button("Recommend karo"):
    results = hybrid_recommend(selected_movie, user_id)
    st.subheader("Tumhare liye recommendations:")
    for i, row in enumerate(results.itertuples(), 1):
        st.write(f"{i}. **{row.title}** — Predicted rating: {row.predicted_rating:.2f}⭐")

Overwriting app.py


In [21]:
!pip install streamlit pyngrok -q
from pyngrok import ngrok

ngrok.set_auth_token("3JLMp69LuTjWQLTW7YGzQkQ1yT2_4PKSTnsvGGvydYaweQeYt")  # ngrok.com se free le lo
!streamlit run app.py &>/content/logs.txt &
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://blog-angles-wharf.ngrok-free.dev" -> "http://localhost:8501"


In [19]:
from pyngrok import ngrok
ngrok.kill()